# NIDS Deep Learning Project Report

            This notebook is the project report layer for the modular NIDS framework.
            It loads experiment artifacts from `results/`, checks for invalid one-class
            runs, and summarizes the supervised, imbalance, anomaly, hybrid, and
            domain-shift findings.

            Recommended final position: use tree-based supervised models such as
            RandomForest/XGBoost for known attacks, then keep anomaly detection as a
            separate suspicious/unknown branch instead of forcing very rare classes
            into ordinary supervised classification.


## 1. Setup

            The report cells below are intentionally data-driven. Re-run the pipeline
            scripts first, then re-run this notebook so tables reflect the current
            `results/` directory.


In [1]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

def load_json(name):
    path = RESULTS_DIR / name
    return json.loads(path.read_text(encoding="utf-8"))

def read_text(name):
    return (RESULTS_DIR / name).read_text(encoding="utf-8")

print("Project root:", PROJECT_ROOT)
print("Results found:", len(list(RESULTS_DIR.glob("*"))))

Project root: G:\My Drive\nids_deep_learning\ids_deep_learning
Results found: 56


## 2. Dataset and Preprocessing

            The project uses adapters for NSL-KDD, UNSW-NB15, and CICIDS2017. The
            corrected preprocessing contract is:

            1. Load and clean raw/cache data.
            2. Map labels.
            3. Reject one-class training data.
            4. Split train/validation/test.
            5. Fit categorical encoders, imputers, and scalers on train only.
            6. Transform validation/test with train-fitted artifacts.

            This order is important because fitting encoders or imputers on the full
            dataset leaks validation/test distribution into training.


In [2]:
for path in sorted(RESULTS_DIR.glob("*_imbalance.json")):
                data = json.loads(path.read_text(encoding="utf-8"))
                print("\n==", path.name, "==")
                if isinstance(data, dict):
                    print(json.dumps(data, indent=2)[:2000])



== cicids2017_multi_imbalance.json ==
{
  "total": 438,
  "classes": [
    {
      "label": "Benign",
      "count": 438,
      "ratio": 1.0
    }
  ]
}

== nsl_kdd_multi_imbalance.json ==
{
  "total": 75583,
  "classes": [
    {
      "label": "Benign",
      "count": 40405,
      "ratio": 0.5345778812695977
    },
    {
      "label": "Buffer Overflow",
      "count": 18,
      "ratio": 0.00023814879007184154
    },
    {
      "label": "DoS",
      "count": 27556,
      "ratio": 0.36457933662331476
    },
    {
      "label": "Ftp Write",
      "count": 5,
      "ratio": 6.615244168662265e-05
    },
    {
      "label": "Guess Passwd",
      "count": 31,
      "ratio": 0.00041014513845706043
    },
    {
      "label": "Probe",
      "count": 6994,
      "ratio": 0.09253403543124777
    },
    {
      "label": "R2L",
      "count": 560,
      "ratio": 0.007409073468901737
    },
    {
      "label": "U2R",
      "count": 14,
      "ratio": 0.00018522683672254342
    }
  ]
}

== uns

## 3. Imbalance Handling

            CICIDS2017 is highly imbalanced. The selected direction is adaptive
            cost-sensitive learning plus rare-class grouping. GAN augmentation is kept
            as legacy/future work because synthetic tabular IDS samples are difficult
            to validate and can increase false alarms.


In [3]:
strategy_path = RESULTS_DIR / "cicids2017_imbalance_strategy.md"
if strategy_path.exists():
    print(strategy_path.read_text(encoding="utf-8"))
else:
    print("Missing cicids2017_imbalance_strategy.md")

# CICIDS2017 Imbalance Strategy

Current training split size: 1,513,416 rows.

The default project direction is now adaptive cost-sensitive learning plus rare grouping. GAN augmentation is not used in the main pipeline because synthetic tabular IDS samples are hard to validate and may increase false alarms.

| Class | Count | Ratio | Strategy |
|---|---:|---:|---|
| Benign | 1,257,890 | 83.1159% | controlled_undersampling |
| DoS | 116,248 | 7.6812% | mild_class_weight |
| DDoS | 76,810 | 5.0753% | mild_class_weight |
| PortScan | 54,491 | 3.6005% | weighted_sampler_plus_class_weight |
| BruteForce | 5,491 | 0.3628% | focal_loss_plus_weighted_sampler |
| WebAttack | 1,285 | 0.0849% | rare_grouping |
| Botnet | 1,172 | 0.0774% | rare_grouping |
| Infiltration | 22 | 0.0015% | rare_grouping |
| Rare_Attack | 7 | 0.0005% | rare_grouping |

## Strategy Bands

| Ratio band | Default strategy | Rationale |
|---|---|---|
| >= 20% | controlled_undersampling | Prevent majority traffic from domi

## 4. Model Comparison: NSL-KDD, UNSW-NB15, CICIDS2017

            The main model selection rule is: prefer higher Macro-F1, then lower FAR,
            then higher minority recall. Accuracy alone is not sufficient for NIDS
            because a majority Benign class can hide poor attack detection.


In [4]:
model_eval_path = RESULTS_DIR / "modular_model_evaluation.csv"
if model_eval_path.exists():
    model_eval = pd.read_csv(model_eval_path)
    display_cols = [
        "dataset", "model", "accuracy", "macro_f1", "weighted_f1",
        "roc_auc", "pr_auc", "far", "minority_recall_mean",
    ]
    display(model_eval[display_cols])
else:
    print("Missing modular_model_evaluation.csv. Run evaluate_modular_results.py first.")

,dataset,model,accuracy,macro_f1,weighted_f1,roc_auc,pr_auc,far,minority_recall_mean
0,cicids2017,RandomForest,0.998545,0.967837,0.998536,0.000000,0.000000,0.000689,0.931664
1,cicids2017,LogisticRegression,0.858494,0.546022,0.905112,0.000000,0.000000,0.168740,0.990124
2,nsl_kdd,XGBoost,0.999047,0.748739,0.998991,0.999903,0.788642,0.000371,0.444444
3,nsl_kdd,LightGBM,0.998889,0.733667,0.998827,0.999928,0.795808,0.000520,0.444444
4,nsl_kdd,RandomForest,0.998254,0.682955,0.998108,0.999809,0.782615,0.000297,0.444444
5,nsl_kdd,CNNLSTMHybrid,0.979520,0.588306,0.982121,0.000000,0.000000,0.033410,0.444444
6,nsl_kdd,CNN1D,0.980790,0.567013,0.983322,0.000000,0.000000,0.031925,0.388889
7,nsl_kdd,MLP,0.974360,0.554910,0.979292,0.000000,0.000000,0.043210,0.444444
8,nsl_kdd,LogisticRegression,0.932129,0.543298,0.943770,0.992080,0.549767,0.111738,0.722222
9,nsl_kdd,BiLSTMAttention,0.945743,0.520063,0.952211,0.000000,0.000000,0.068305,0.388889


In [5]:
per_class_path = RESULTS_DIR / "modular_per_class_evaluation.csv"
if per_class_path.exists():
    per_class = pd.read_csv(per_class_path)
    low_support = per_class.sort_values(["dataset", "support", "model"]).head(30)
    display(low_support)
else:
    print("Missing modular_per_class_evaluation.csv.")

,dataset,model,label,support,precision,recall,f1
7,cicids2017,LogisticRegression,Rare_Attack,2,0.400000,1.000000,0.571429
16,cicids2017,RandomForest,Rare_Attack,2,1.000000,1.000000,1.000000
5,cicids2017,LogisticRegression,Infiltration,7,0.001162,1.000000,0.002322
14,cicids2017,RandomForest,Infiltration,7,1.000000,0.857143,0.923077
1,cicids2017,LogisticRegression,Botnet,391,0.012052,0.982097,0.023812
10,cicids2017,RandomForest,Botnet,391,0.876833,0.764706,0.816940
8,cicids2017,LogisticRegression,WebAttack,429,0.027446,0.990676,0.053412
17,cicids2017,RandomForest,WebAttack,429,0.990610,0.983683,0.987135
2,cicids2017,LogisticRegression,BruteForce,1830,0.503834,0.969399,0.663054
11,cicids2017,RandomForest,BruteForce,1830,0.999451,0.995628,0.997536


## 5. NSL-KDD Findings

            NSL-KDD is useful for fast debugging and model comparison. The stronger
            classical ML models generally outperform the deep models in Macro-F1 and
            FAR on the current modular results.


In [6]:
nsl_path = RESULTS_DIR / "nsl_kdd_multi_modular_results.json"
if nsl_path.exists():
    nsl = load_json("nsl_kdd_multi_modular_results.json")
    rows = []
    for model, metrics in nsl["models"].items():
        rows.append({
            "model": model,
            "accuracy": metrics.get("accuracy"),
            "macro_f1": metrics.get("macro_f1"),
            "far": metrics.get("far"),
            "roc_auc": metrics.get("roc_auc"),
            "pr_auc": metrics.get("pr_auc"),
        })
    display(pd.DataFrame(rows).sort_values("macro_f1", ascending=False))

,model,accuracy,macro_f1,far,roc_auc,pr_auc
6,XGBoost,0.999047,0.748739,0.000371,0.999903,0.788642
7,LightGBM,0.998889,0.733667,0.000520,0.999928,0.795808
1,RandomForest,0.998254,0.682955,0.000297,0.999809,0.782615
5,CNNLSTMHybrid,0.979520,0.588306,0.033410,NaN,NaN
3,CNN1D,0.980790,0.567013,0.031925,NaN,NaN
2,MLP,0.974360,0.554910,0.043210,NaN,NaN
0,LogisticRegression,0.932129,0.543298,0.111738,0.992080,0.549767
4,BiLSTMAttention,0.945743,0.520063,0.068305,NaN,NaN


## 6. CICIDS2017 Validity Check

            CICIDS2017 results must be checked carefully. A model result with a 1x1
            confusion matrix or only one active class is invalid for multi-class
            evaluation, even if it reports 100% accuracy.


In [7]:
def invalid_reason(metrics):
    confusion = metrics.get("confusion_matrix", [])
    report = metrics.get("classification_report", {})
    class_rows = [v for k, v in report.items() if str(k).isdigit()]
    active = [row for row in class_rows if float(row.get("support", 0)) > 0]
    if len(confusion) < 2 or len(active) < 2:
        return "degenerate_one_class"
    return ""

for path in sorted(RESULTS_DIR.glob("*_modular_results.json")):
    result = json.loads(path.read_text(encoding="utf-8"))
    invalid = []
    for model, metrics in result.get("models", {}).items():
        reason = invalid_reason(metrics)
        if reason:
            invalid.append({"dataset": result.get("dataset"), "model": model, "reason": reason})
    if invalid:
        print("\nInvalid results in", path.name)
        display(pd.DataFrame(invalid))


Invalid results in cicids2017_multi_BiLSTMAttention_Modular_results.json


,dataset,model,reason
0,cicids2017,BiLSTMAttention_Modular,degenerate_one_class



Invalid results in cicids2017_multi_CNN1D_Modular_results.json


,dataset,model,reason
0,cicids2017,CNN1D_Modular,degenerate_one_class



Invalid results in cicids2017_multi_CNNLSTMHybrid_Modular_results.json


,dataset,model,reason
0,cicids2017,CNNLSTMHybrid_Modular,degenerate_one_class



Invalid results in cicids2017_multi_MLP_Modular_results.json


,dataset,model,reason
0,cicids2017,MLP_Modular,degenerate_one_class



Invalid results in cicids2017_multi_modular_results.json


,dataset,model,reason
0,cicids2017,MLP,degenerate_one_class
1,cicids2017,CNN1D,degenerate_one_class
2,cicids2017,BiLSTMAttention,degenerate_one_class
3,cicids2017,CNNLSTMHybrid,degenerate_one_class


## 7. Domain Shift

            Cross-dataset tests estimate how much performance changes when training
            and testing distributions differ. Large drops in Macro-F1 and increases
            in FAR support the conclusion that supervised NIDS models do not transfer
            cleanly across network environments without adaptation.


In [8]:
report_path = RESULTS_DIR / "cross_dataset_report.md"
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Missing cross_dataset_report.md")

# Báo cáo thực nghiệm Cross-Dataset & Domain Shift

Báo cáo này đánh giá khả năng tổng quát hóa của các mô hình phát hiện xâm nhập khi được huấn luyện trên một tập dữ liệu mạng và kiểm thử chéo trên một tập dữ liệu mạng khác hoàn toàn.

## Không gian đặc trưng cốt lõi chung
- `duration`: thời lượng luồng mạng
- `src_bytes`: số byte truyền từ nguồn đến đích
- `dst_bytes`: số byte truyền từ đích đến nguồn
- `count`: số lượng kết nối tương thích

## Kết quả Đánh giá chéo

| Kịch bản | Mô hình | Self Acc | Self Macro-F1 | Cross Acc | Cross Macro-F1 | Cross FAR |
|---|---|---:|---:|---:|---:|---:|
| NSL-KDD &rarr; UNSW-NB15 | RandomForest | 0.980655 | 0.980538 | 0.583321 | 0.519861 | 0.755486 |
| UNSW-NB15 &rarr; NSL-KDD | RandomForest | 0.996186 | 0.996148 | 0.530201 | 0.362019 | 0.023878 |
| NSL-KDD &rarr; UNSW-NB15 | XGBoost | 0.979369 | 0.979244 | 0.530729 | 0.451980 | 0.831270 |
| UNSW-NB15 &rarr; NSL-KDD | XGBoost | 0.958898 | 0.958604 | 0.242076 | 0.238589 | 0.710289 |

## Thảo luận 

## 8. Anomaly Detection

            The anomaly branch is trained on Benign-only traffic and tuned by target
            FAR. Its role is suspicious/unknown detection, not replacing the known
            attack classifier for labels that are already well learned.


In [9]:
for path in sorted(RESULTS_DIR.glob("*IsolationForest*results.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for threshold, metrics in data.get("evaluations", {}).items():
        row = {"threshold": threshold}
        row.update(metrics)
        rows.append(row)
    print("\n==", path.name, "==")
    display(pd.DataFrame(rows))


== cicids2017_multi_IsolationForest_BenignOnly_10pct_results.json ==


,threshold,accuracy,precision,recall,f1,far,attack_recall,n_alerts,n_samples
0,0.146201,0.840271,0.674772,0.104250,0.180598,0.010207,0.104250,1316,50448
1,0.069004,0.873454,0.729759,0.397863,0.514967,0.029931,0.397863,4644,50448
2,0.033968,0.857140,0.618087,0.402794,0.487739,0.050560,0.402794,5551,50448
3,-0.042441,0.844969,0.537277,0.589692,0.562266,0.103172,0.589692,9349,50448



== cicids2017_multi_IsolationForest_BenignOnly_25pct_results.json ==


,threshold,accuracy,precision,recall,f1,far,attack_recall,n_alerts,n_samples
0,0.149162,0.839612,0.669962,0.098666,0.172002,0.009874,0.098666,3136,126119
1,0.074292,0.874230,0.736998,0.396638,0.515723,0.028753,0.396638,11460,126119
2,0.038954,0.858253,0.624227,0.403165,0.489914,0.049301,0.403165,13753,126119
3,-0.042579,0.847541,0.544491,0.593688,0.568027,0.100892,0.593688,23218,126119


## 9. Hybrid Decision

            Hybrid results should be interpreted by the trade-off between recovered
            missed attacks and extra false alarms. For CICIDS2017, the tested
            RandomForest OR IsolationForest rule was rejected for known labels because
            it increased FAR without recovering RandomForest's missed attacks.


In [10]:
for path in sorted(RESULTS_DIR.glob("*hybrid*results.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\n==", path.name, "==")
    keys = [key for key in ["classifier_alone", "rf_multiclass", "if_binary", "hybrid_binary"] if key in data]
    for key in keys:
        print(key)
        print(json.dumps(data[key], indent=2)[:2000])


== cicids2017_multi_CNNLSTMHybrid_Modular_results.json ==

== cicids2017_multi_Hybrid_RF_IF_inline_10pct_results.json ==
rf_multiclass
{
  "accuracy": 0.9982159847764034,
  "precision_weighted": 0.9982097530016535,
  "recall_weighted": 0.9982159847764034,
  "macro_f1": 0.9732446099779436,
  "weighted_f1": 0.9982121955710932,
  "far": 0.0008347245409015025,
  "confusion_matrix": [
    [
      41895,
      8,
      1,
      1,
      9,
      0,
      16,
      0,
      0
    ],
    [
      9,
      30,
      0,
      0,
      0,
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      183,
      0,
      0,
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      2560,
      0,
      0,
      0,
      0,
      0
    ],
    [
      17,
      0,
      0,
      0,
      3858,
      0,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      0,
      0,
      1,
      0,
      0,
      0
    ],
    [
      27,
      0,
      0,
      0,



== nsl_kdd_hybrid_decision_results.json ==
classifier_alone
{
  "accuracy": 0.9982933121651121,
  "precision": 0.9996578564707895,
  "recall": 0.9966740576496674,
  "f1": 0.9981637272067302,
  "far": 0.00029697824634345533,
  "attack_recall": 0.9966740576496674
}

== nsl_kdd_multi_CNNLSTMHybrid_Modular_results.json ==



== nsl_kdd_multi_CNNLSTMHybrid_Smoke_results.json ==



== unsw_nb15_hybrid_decision_results.json ==
classifier_alone
{
  "accuracy": 0.9598713021717759,
  "precision": 0.9807644882860665,
  "recall": 0.9146734130634775,
  "f1": 0.9465667023682018,
  "far": 0.01140184183598889,
  "attack_recall": 0.9146734130634775
}

== unsw_nb15_multi_CNNLSTMHybrid_Modular_results.json ==


## 10. Final Conclusion

            The strongest current story is not "deep learning wins everywhere." The
            stronger and more defensible conclusion is:

            - Use RandomForest/XGBoost/LightGBM as robust known-attack baselines.
            - Use Macro-F1, minority recall, and FAR as primary IDS metrics.
            - Treat very rare or unseen behavior as an anomaly/suspicious branch.
            - Keep GAN and over-aggressive sampling as future work unless false alarm
              impact is controlled.
            - Re-run CICIDS2017 only with a verified multi-class raw/cache dataset.
